# 04. Product Performance & Priority
**Enterprise Retail Intelligence & Decision Engine**  
*Phase 8: Python Exploratory Data Analysis & Business Intelligence*

---

### Overview & Objectives
Top SKUs, reorder rates, cart-insertion positions, and Pareto distribution.

---


## 1. Setup & Data Loading

Load product catalogs and order-product interactions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path("../data/Processed")
df_products = pd.read_parquet(DATA_DIR / "products_clean.parquet")
df_depts = pd.read_parquet(DATA_DIR / "departments_clean.parquet")
df_aisles = pd.read_parquet(DATA_DIR / "aisles_clean.parquet")
df_train = pd.read_parquet(DATA_DIR / "order_products_train_clean.parquet")

print("Products and Train interactions loaded.")

## 2. Top 20 Most Ordered Products

Rank products by total order volume in the evaluation dataset.

In [ ]:
prod_counts = df_train['product_id'].value_counts().reset_index()
prod_counts.columns = ['product_id', 'order_volume']
top20 = prod_counts.head(20).merge(df_products, on='product_id')

plt.figure(figsize=(12, 6))
sns.barplot(data=top20, y='product_name', x='order_volume', palette='viridis')
plt.title("Top 20 Most Ordered Products (Instacart Train Set)")
plt.xlabel("Order Volume")
plt.ylabel("Product Name")
plt.tight_layout()
plt.show()
top20[['product_name', 'order_volume']]

## 3. Reorder Propensity by Product

Examine which products have the highest reorder rates (minimum 500 orders).

In [ ]:
reorder_stats = df_train.groupby('product_id').agg(
    total_orders=('reordered', 'count'),
    reorders=('reordered', 'sum')
).reset_index()
reorder_stats['reorder_rate'] = reorder_stats['reorders'] / reorder_stats['total_orders']

# Filter to products with substantial volume
popular_reorders = reorder_stats[reorder_stats['total_orders'] >= 500].sort_values(by='reorder_rate', ascending=False)
popular_reorders = popular_reorders.head(15).merge(df_products, on='product_id')

plt.figure(figsize=(10, 5))
sns.barplot(data=popular_reorders, y='product_name', x='reorder_rate', palette='Blues_r')
plt.title("Top Products by Reorder Rate (Min 500 Orders)")
plt.xlabel("Reorder Rate (Proportion)")
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()

## 4. Add-to-Cart Priority Analysis

Analyze which items are placed in the cart first (impulse vs essential pantry staples).

In [ ]:
first_items = df_train[df_train['add_to_cart_order'] == 1]['product_id'].value_counts().head(10).reset_index()
first_items.columns = ['product_id', 'first_cart_count']
first_items = first_items.merge(df_products, on='product_id')

print("--- Top 10 First-Added Products to Cart ---")
print(first_items[['product_name', 'first_cart_count']])